In [1]:
import os
import sys
from pathlib import Path
from typing import final

# If you need to load .env file in notebook
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from src.kg.neo4j_client import get_neo4j_database, get_neo4j_driver
from src.rag.retrieve import retrieve_candidates_graph
from src.rag.merge import merge_candidates
from src.rag.planner import plan_query
from src.rag.composer import compose_answer

In [2]:
# --- Configuration ---
PROJECT_ROOT = Path(".").resolve().parent

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Missing OPENAI_API_KEY env var.")

openai_client = OpenAI(api_key=api_key)
driver = get_neo4j_driver()
database = get_neo4j_database()

# Configs
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-large")
INDEX_NAME = os.getenv("NEO4J_VECTOR_INDEX", "offer_embedding_index")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o")

### Testing

In [3]:
user_query = "ბათუმში მივდივარ ივენთზე ,მჭირდება ტანსაცმელი"

print(f"🔎 Analyzing: '{user_query}'...")

# PLAN
plan = plan_query(openai_client, user_query, PROJECT_ROOT)
print(f"📋 Plan: {plan}")

🔎 Analyzing: 'ბათუმში მივდივარ ივენთზე ,მჭირდება ტანსაცმელი'...
📋 Plan: cities=['ბათუმი'] categories=['შოპინგი'] segment_types=[] product_codes=[] rewritten_queries=['ტანსაცმლის მაღაზიები ბათუმში', 'ივენთისთვის შესაფერისი ტანსაცმელი ბათუმში', 'შოპინგი ბათუმში']


In [4]:
# RETRIEVE
all_hits = []
for rq in plan.rewritten_queries:
    hits = retrieve_candidates_graph(
        driver=driver,
        database=database,
        openai_client=openai_client,
        embedding_model=EMBEDDING_MODEL,
        index_name=INDEX_NAME,
        query=rq,
        top_k=12,
        cities=plan.cities,
        categories=plan.categories,
        segment_types=plan.segment_types,
        product_codes=plan.product_codes,
    )
    all_hits.extend(hits)
    

In [5]:
# MERGE & RANK
merged = merge_candidates(all_hits)
final_offers = merged[:3]

In [6]:
len(merged), len(final_offers)

(14, 3)

In [7]:
print(f"DEBUG: Sending {len(final_offers)} offers to LLM.")
for o in final_offers:
    print(f" - {o.title} (Relaxed: {o.is_relaxed}),  Cities: {o.cities}, (Category: {o.category}) , (Score: {o.score}), ")


DEBUG: Sending 3 offers to LLM.
 - ბათუმი (Relaxed: False),  Cities: ბათუმი, (Category: შოპინგი) , (Score: 0.793087100982666), 
 - Manga Mania (Relaxed: False),  Cities: თბილისი, ბათუმი, (Category: შოპინგი) , (Score: 0.7630187034606933), 
 - Pendant (Relaxed: False),  Cities: თბილისი, ბათუმი, რუსთავი, (Category: შოპინგი) , (Score: 0.7596908569335937), 


In [8]:
final_offers[2]

RetrievedOffer(campaign_id=14011, title='Pendant', category='შოპინგი', short_desc='10-ჯერ მეტი PLUS', cities='თბილისი, ბათუმი, რუსთავი', long_desc='თებერვლის ბოლომდე შეიძინე სამკაულები Pendant-ში, გადაიხადე PLUS ბარათით და დააგროვე 10-ჯერ მეტი PLUS ქულა. Pendant არის აქსესუარების, ბიჟუტერიისა და ხელნაკეთი ნივთების მაღაზია. აქსესუარები მზადდება როდირებული ვერცხლით. მისამართი: სითი მოლი საბურთალო სითი მოლი გლდანი გალერია თბილისი გლდანი მოლი ისთ ფოინთი ბათუმი - შავი ზღვის მოლი რუსთავი მოლი', brand_name='პენდანტი', segment_types='RETAIL, SOLO, WM', product_codes='PLUS', score=0.7596908569335937, is_relaxed=False)

In [11]:
# GENERATE
print(f"🤖 Generating answer based on {len(final_offers)} offers...")
answer = compose_answer(
    client=openai_client,
    user_query=user_query,
    offers=final_offers,
    model=CHAT_MODEL,
)

🤖 Generating answer based on 3 offers...


In [12]:
print(answer)

გამარჯობა, მე ვარ საქართველოს ბანკის შეთავაზებების ასისტენტი და მზად ვარ დაგეხმაროთ საუკეთესო შეთავაზებების პოვნაში. ბათუმში ივენთზე წასვლისთვის ტანსაცმლის შეძენას გეგმავთ? შესანიშნავია! 

თუ PLUS ბარათის მფლობელი ხართ, ლუტეციაში შესანიშნავი შესაძლებლობა გაქვთ: 10X PLUS ქულა! ეს ნიშნავს, რომ თქვენი ყოველ შესყიდვაზე მიიღებთ ათჯერ მეტ ქულას, რაც ნამდვილად გამოგადგებათ მომავალი შესყიდვებისთვის. მეტი დეტალისთვის ეწვიეთ: https://bankofgeorgia.ge/ka/offers-hub/details/15045

ასევე, პენდანტში, რომელიც ბათუმშიც მდებარეობს, შეგიძლიათ მიიღოთ 10-ჯერ მეტი PLUS ქულა. ეს არის შესანიშნავი გზა, რომ გაამრავლოთ თქვენი ბონუსები და ისარგებლოთ დამატებითი სარგებელით. მეტი დეტალისთვის ეწვიეთ: https://bankofgeorgia.ge/ka/offers-hub/details/14011

თუ სტუდენტი ხართ ან გაქვთ SCHOOL_CARD, მინისოში შეგიძლიათ მიიღოთ 20% ქეშბექი. ეს არის შესანიშნავი შეთავაზება, რომელიც დაგეხმარებათ დაზოგოთ თანხა და ისარგებლოთ ხარისხიანი პროდუქციით. მეტი დეტალისთვის ეწვიეთ: https://bankofgeorgia.ge/ka/offers-hub/details/11400

იმედია